In [ ]:
# Imports etc.
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway, kruskal, levene, ttest_ind
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
# Load the data
root = Path('/media/yannik/Intenso/DATA/dfg_plexus')
ms_data = pd.read_csv(root / "Dataset002_DFGFinetuned_3d_fullres_full_ms___volumes.csv")
ms_metadata = pd.read_excel(root / "MSData_noNMO.xlsx")
hc_data = pd.read_csv(root / "Dataset002_DFGFinetuned_3d_fullres_full_new_hc___volumes.csv")
hc_metadata = pd.read_excel(root / "HCData.xlsx")

In [ ]:
# Adjust the column name for the patient ID (replace 'actual_patient_id_column' with the correct name)
ms_metadata_id_column = "patID"
hc_metadata_id_column = "subID"
data_id_column = "subject_id"

In [ ]:
# Merge the volume data with metadata
patho_merged_data = ms_metadata.merge(ms_data[['subject_id', 'volume']], left_on=ms_metadata_id_column, right_on=data_id_column)
hc_merged_data = hc_metadata.merge(hc_data[['subject_id', 'volume']], left_on=hc_metadata_id_column, right_on=data_id_column)
merged_data = pd.concat([patho_merged_data, hc_merged_data], axis=0)

In [ ]:
print(merged_data)

### Hypothesis Testing
Null Hypothesis: The mean volume of the choroid plexus is the same for all genders. <br>
Alternative Hypothesis: The mean volume of the choroid plexus is different across genders.

In [ ]:
# Check group statistics
grouped = merged_data.groupby("Sex")['volume']
#grouped = hc_merged_data.groupby("Sex")['volume']
#grouped = patho_merged_data.groupby("Sex")['volume']
group_stats = grouped.describe()
print(group_stats)

In [ ]:
# Visualize data
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=merged_data,
    # data=hc_merged_data,
    # data=patho_merged_data,
    x="Sex",
    y="volume",
    palette="Set2",
    hue="Sex"
)
plt.title("Choroid Plexus Volume by Gender")
plt.xlabel("Gender")
plt.ylabel("Volume")
plt.xticks(rotation=45)
# plt.savefig("volume_by_gender_hc.png")
# plt.savefig("volume_by_gender_patho.png")
plt.show()

In [ ]:
# Check for homogeneity of variances with Levene's test
levene = levene(*[group[1] for group in grouped])
print("Levene's Test for Homogeneity of Variances:")
print(f"W={levene.statistic}, p={levene.pvalue}")

In [ ]:
# Test for statistical significance
ttest = ttest_ind(*[group[1] for group in grouped], equal_var=False if levene.statistic < 0.05 else True)
print("Two-Sample T-Test:")
print(f"Statistic={ttest.statistic}, p={ttest.pvalue}")

In [ ]:
# Effect size calculation
mean_diff = group_stats['mean'].diff().abs().iloc[-1]

def cliffs_delta(x, y):
    n_x = len(x)
    n_y = len(y)
    pairs = [(xi, yi) for xi in x for yi in y]
    greater = sum(xi > yi for xi, yi in pairs)
    smaller = sum(xi < yi for xi, yi in pairs)
    return (greater - smaller) / (n_x * n_y)

delta = cliffs_delta(*[group[1] for group in grouped])
print(f"Cliff's Delta: {delta:.3f}")

In [ ]:
# Interpret Cliff's Delta
if abs(delta) < 0.147:
    interpretation = "negligible"
elif abs(delta) < 0.33:
    interpretation = "small"
elif abs(delta) < 0.474:
    interpretation = "medium"
else:
    interpretation = "large"

print(f"Effect size interpretation: {interpretation}")